# Tracking Training Speedup with Callbacks (scikit-learn)

scikit-learn estimators have no callback hooks, but those that implement `partial_fit` can be trained batch by batch from your own loop. This notebook builds a tiny callback mechanism for that, and uses it with an `MLPClassifier` to:

- measure the training throughput (samples per second) and the speedup over the first epoch
- show progress with psnek: one bar per epoch, plus a table of finished epochs in the panel

Training speeds up because the batch size doubles every epoch, from 16 to 1024 samples, as larger batches amortize the overhead of each `partial_fit` call. The validation score is tracked too, to confirm the faster epochs don't hurt the model.


In [1]:
import time
from collections.abc import Sequence
from dataclasses import dataclass, field

import numpy as np
import polars as pl
from sklearn.base import BaseEstimator, is_classifier
from sklearn.datasets import make_classification
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier

import psnek

A synthetic four-class problem with 120,000 samples and 40 features.


In [2]:
features, labels = make_classification(
    n_samples=120_000, n_features=40, n_informative=24, n_classes=4, random_state=0
)
x_train, x_val, y_train, y_val = train_test_split(
    features, labels, test_size=0.2, random_state=0
)

## The Callback Mechanism

`fit()` only needs `partial_fit` and `score`, so it works with any incremental estimator. Every callback hook receives a shared `Run` with the training state, and callbacks can add columns to each epoch's row in `run.history`.


In [3]:
@dataclass
class Run:
    """Training progress shared with every callback."""

    model: BaseEstimator
    epochs: int
    samples_per_epoch: int
    history: list[dict[str, float]] = field(default_factory=list)
    epoch: int = 0
    batch_size: int = 0
    samples: int = 0
    seconds: float = 0.0

    @property
    def metrics(self) -> dict[str, float]:
        """The latest epoch's row of ``history``, which callbacks may extend."""
        return self.history[-1]


class Callback:
    """Hooks called by `fit` with the shared `Run`; override the ones you need."""

    def on_train_begin(self, run: Run) -> None:
        """Called once before the first epoch."""

    def on_epoch_begin(self, run: Run) -> None:
        """Called before the first batch of an epoch."""

    def on_batch_end(self, run: Run) -> None:
        """Called after each `partial_fit` step."""

    def on_epoch_end(self, run: Run) -> None:
        """Called after an epoch, once its metrics row is in the history."""

    def on_train_end(self, run: Run) -> None:
        """Called once after the last epoch."""


def fit(
    model: BaseEstimator,
    train: tuple[np.ndarray, np.ndarray],
    val: tuple[np.ndarray, np.ndarray],
    schedule: Sequence[int],
    callbacks: Sequence[Callback],
) -> list[dict[str, float]]:
    """Train ``model`` with one shuffled epoch per batch size in ``schedule``.

    Args:
        model: Any estimator with ``partial_fit`` and ``score``.
        train: Training features and targets.
        val: Validation features and targets, scored after each epoch.
        schedule: Batch size of each epoch.
        callbacks: Hooks run in order at each stage.

    Returns:
        One metrics row per epoch.
    """
    features, targets = train
    run = Run(model, len(schedule), len(features))
    extra = {"classes": np.unique(targets)} if is_classifier(model) else {}
    tunable = "batch_size" in model.get_params()
    rng = np.random.default_rng(0)

    def emit(hook: str) -> None:
        for callback in callbacks:
            getattr(callback, hook)(run)

    emit("on_train_begin")
    for epoch, batch_size in enumerate(schedule, start=1):
        run.epoch, run.batch_size, run.samples, run.seconds = epoch, batch_size, 0, 0.0
        emit("on_epoch_begin")
        order = rng.permutation(len(features))
        start = time.perf_counter()
        for rows in np.split(order, range(batch_size, len(order), batch_size)):
            if tunable and model.batch_size != len(rows):
                model.set_params(batch_size=len(rows))
            model.partial_fit(features[rows], targets[rows], **extra)
            run.samples += len(rows)
            run.seconds = time.perf_counter() - start
            emit("on_batch_end")
        score = round(model.score(*val), 4)
        run.history.append(
            {"epoch": epoch, "batch_size": batch_size, "val_score": score}
        )
        emit("on_epoch_end")
    emit("on_train_end")
    return run.history

`SpeedTracker` computes each epoch's throughput and speedup. `PsnekProgress` draws a bar per epoch that counts samples, so its rate is the live throughput, and shows the history table in the panel.


In [4]:
class SpeedTracker(Callback):
    """Add throughput and the speed-up over the first epoch to each epoch's row."""

    def on_epoch_end(self, run: Run) -> None:
        """Add ``samples_per_s`` and ``speedup`` to the epoch's metrics."""
        run.metrics["samples_per_s"] = round(run.samples / run.seconds)
        baseline = run.history[0]["samples_per_s"]
        run.metrics["speedup"] = round(run.metrics["samples_per_s"] / baseline, 2)


class PsnekProgress(Callback):
    """Show training with psnek: an epochs bar, a kept bar per epoch, and a table."""

    epochs: psnek.Bar
    current: psnek.Bar

    def on_train_begin(self, run: Run) -> None:
        """Open the top-level epochs bar."""
        self.epochs = psnek.Bar(total=run.epochs, desc="epochs")

    def on_epoch_begin(self, run: Run) -> None:
        """Open a kept bar counting this epoch's samples."""
        self.current = psnek.Bar(
            total=run.samples_per_epoch,
            desc=f"epoch {run.epoch} (batch {run.batch_size:>4})",
            options=psnek.Options(unit="", scale="si", leave=True),
        )

    def on_batch_end(self, run: Run) -> None:
        """Advance the epoch bar to the samples seen."""
        self.current.update(run.samples - self.current.n)

    def on_epoch_end(self, run: Run) -> None:
        """Close the epoch bar and refresh the history table."""
        self.current.close()
        self.epochs.update()
        self.epochs.panel(run.history)

    def on_train_end(self, run: Run) -> None:
        """Close the epochs bar."""
        self.epochs.close()

## Train


In [5]:
schedule = [16, 32, 64, 128, 256, 512, 1024]
callbacks = [SpeedTracker(), PsnekProgress()]
model = MLPClassifier(
    hidden_layer_sizes=(128, 64), learning_rate_init=1e-3, random_state=0
)
history = fit(model, (x_train, y_train), (x_val, y_val), schedule, callbacks)

# epoch 1 (batch   16)   ##############  100.0%  96.0k/96.0k         0:05    19.1k/s  done
# epoch 2 (batch   32)   ##############  100.0%  96.0k/96.0k         0:02    41.6k/s  done
# epoch 3 (batch   64)   ##############  100.0%  96.0k/96.0k         0:01    72.3k/s  done
# epoch 4 (batch  128)   ##############  100.0%  96.0k/96.0k         0:00     125k/s  done
# epoch 5 (batch  256)   ##############  100.0%  96.0k/96.0k         0:00     200k/s  done
# epoch 6 (batch  512)   ##############  100.0%  96.0k/96.0k         0:00     336k/s  done
# epoch 7 (batch 1024)   ##############  100.0%  96.0k/96.0k         0:00     450k/s  done
# epochs                 ##############  100.0%          7/7         0:10   1.45s/it  done
| epoch  batch_size  val_score  samples_per_s  speedup
| 1      16          0.9609     19140          1.000
| 2      32          0.9703     41759          2.180
| 3      64          0.9744     72864          3.810
| 4      128         0.9789     127142         6.640
| 5      256         0.9805     204620         10.69
| 6      512         0.9816     347985         18.18
| 7      1024        0.9814     472530         24.69

## Results

The history `fit()` returned, as a polars DataFrame.


In [6]:
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)
mlp_history = pl.DataFrame(history)
print(mlp_history)

┌───────┬────────────┬───────────┬───────────────┬─────────┐
│ epoch ┆ batch_size ┆ val_score ┆ samples_per_s ┆ speedup │
╞═══════╪════════════╪═══════════╪═══════════════╪═════════╡
│ 1     ┆ 16         ┆ 0.9609    ┆ 19140         ┆ 1.0     │
│ 2     ┆ 32         ┆ 0.9703    ┆ 41759         ┆ 2.18    │
│ 3     ┆ 64         ┆ 0.9744    ┆ 72864         ┆ 3.81    │
│ 4     ┆ 128        ┆ 0.9789    ┆ 127142        ┆ 6.64    │
│ 5     ┆ 256        ┆ 0.9805    ┆ 204620        ┆ 10.69   │
│ 6     ┆ 512        ┆ 0.9816    ┆ 347985        ┆ 18.18   │
│ 7     ┆ 1024       ┆ 0.9814    ┆ 472530        ┆ 24.69   │
└───────┴────────────┴───────────┴───────────────┴─────────┘


In [7]:
final = mlp_history.row(-1, named=True)
print(
    f"throughput rose {final['speedup']:.1f}x, from "
    f"{mlp_history['samples_per_s'][0]:,} to {final['samples_per_s']:,} samples/s, "
    f"while validation accuracy went from {mlp_history['val_score'][0]:.2%} "
    f"to {final['val_score']:.2%}"
)

throughput rose 24.7x, from 19,140 to 472,530 samples/s, while validation accuracy went from 96.09% to 98.14%


The same loop and callbacks work unchanged with an `SGDClassifier`. Its speedup is even larger, since each call does less work, although a linear model can't fit this problem as well.


In [8]:
model = SGDClassifier(loss="log_loss", random_state=0)
history = fit(model, (x_train, y_train), (x_val, y_val), schedule, callbacks)

# epoch 1 (batch   16)   ##############  100.0%  96.0k/96.0k         0:04    21.4k/s  done
# epoch 2 (batch   32)   ##############  100.0%  96.0k/96.0k         0:02    41.9k/s  done
# epoch 3 (batch   64)   ##############  100.0%  96.0k/96.0k         0:01    82.4k/s  done
# epoch 4 (batch  128)   ##############  100.0%  96.0k/96.0k         0:00     159k/s  done
# epoch 5 (batch  256)   ##############  100.0%  96.0k/96.0k         0:00     300k/s  done
# epoch 6 (batch  512)   ##############  100.0%  96.0k/96.0k         0:00     490k/s  done
# epoch 7 (batch 1024)   ##############  100.0%  96.0k/96.0k         0:00     763k/s  done
# epochs                 ##############  100.0%          7/7         0:09   1.31s/it  done
| epoch  batch_size  val_score  samples_per_s  speedup
| 1      16          0.4810     21372          1.000
| 2      32          0.4986     41913          1.960
| 3      64          0.4517     82615          3.870
| 4      128         0.5174     159702         7.470
| 5      256         0.4938     302346         14.15
| 6      512         0.4925     497331         23.27
| 7      1024        0.5063     781276         36.56